In [1]:

import os, gc, glob, warnings
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
warnings.filterwarnings('ignore')

SAVE_DIR  = '/srv/work'
IN_LABEL  = f'{SAVE_DIR}/te_labeled_ventanas_v3.parquet'
IN_BASE   = f'{SAVE_DIR}/te_baseline_stats_v3.parquet'
ARCHIVOS  = sorted(
    glob.glob(f'{SAVE_DIR}/te_*_raw_may_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_w[0-9].parquet')
)
OUT_FEAT  = f'{SAVE_DIR}/te_features_v4.parquet'

METRICS_NET = ['network.latency', 'network.loss', 'network.jitter']
METRICS_BGP = ['bgp.path_changes.count', 'bgp.updates.count', 'bgp.reachability']

METRIC_SHORT = {
    'network.latency': 'lat',
    'network.loss':    'loss',
    'network.jitter':  'jitter',
}

BD_END  = pd.Timestamp('2026-05-08', tz='UTC')
EPS     = 1e-9
WINDOW  = pd.Timedelta('30min')
H       = 60
W_v3    = 30
W_v4    = 90

print(f'Archivos raw : {len(ARCHIVOS)}  (esperado: 39)')
print(f'Labels       : {IN_LABEL}')
print(f'Baseline     : {IN_BASE}')
print(f'Output       : {OUT_FEAT}')
print(f'W_v3={W_v3}min  H={H}min  W_v3 < H  PROBLEMA (LOG-10)')
print(f'W_v4={W_v4}min  H={H}min  W_v4 >= H CORREGIDO (lag t-1/t-2/t-48/t-336)')

Archivos raw : 39  (esperado: 39)
Labels       : /srv/work/te_labeled_ventanas_v3.parquet
Baseline     : /srv/work/te_baseline_stats_v3.parquet
Output       : /srv/work/te_features_v4.parquet
W_v3=30min  H=60min  W_v3 < H  PROBLEMA (LOG-10)
W_v4=90min  H=60min  W_v4 >= H CORREGIDO (lag t-1/t-2/t-48/t-336)


In [2]:
labeled_df = pd.read_parquet(IN_LABEL)
labeled_df['window'] = pd.to_datetime(labeled_df['window'], utc=True)
print(f'Labels v3 shape : {labeled_df.shape}')
print(f'Splits:')
print(labeled_df['split'].value_counts())
print(f'\nDistribucion global:')
dist = labeled_df['label_ventana'].value_counts(normalize=True).sort_index() * 100
labels_map = {0:'Normal', 1:'Baja', 2:'Media', 3:'Alta'}
for k, pct in dist.items():
    print(f'  {labels_map[k]:8s} ({k}): {pct:.1f}%')

baseline_df = pd.read_parquet(IN_BASE)
print(f'\nBaseline v3 shape: {baseline_df.shape}')
print(baseline_df.groupby('metric')[['p95','mad']].mean().round(4))

baseline = {}
for row in baseline_df.itertuples():
    key = (row.source_agent, row.cloud_provider, row.region, row.metric)
    baseline[key] = {'p95': row.p95, 'mad': row.mad}
print(f'\nBaseline entradas: {len(baseline):,}  (esperado 3,105 = 1,035 rutas x 3 metricas)')

Labels v3 shape : (3763917, 6)
Splits:
split
train    2125778
test      993449
val       644690
Name: count, dtype: int64

Distribucion global:
  Normal   (0): 87.8%
  Baja     (1): 8.1%
  Media    (2): 2.0%
  Alta     (3): 2.1%

Baseline v3 shape: (3105, 6)
                    p95     mad
metric                         
network.jitter   2.3384  0.4655
network.latency  0.1575  0.0078
network.loss     4.6274  0.5000

Baseline entradas: 3,105  (esperado 3,105 = 1,035 rutas x 3 metricas)


In [3]:
PROVIDER_MAP = {
    'aws':   [f for f in ARCHIVOS if '/te_aws_'   in f.lower()],
    'azure': [f for f in ARCHIVOS if '/te_azure_' in f.lower()],
    'gcp':   [f for f in ARCHIVOS if '/te_gcp_'   in f.lower()],
}

net_records = []

for prov, files in PROVIDER_MAP.items():
    print(f'\n[{prov.upper()}] {len(files)} archivos...')
    for fi, f in enumerate(files):
        tbl = pq.read_table(f, columns=['_time','source_agent','cloud_provider','region','metric','value'])
        df  = tbl.to_pandas(); del tbl; gc.collect()

        df['_time'] = pd.to_datetime(df['_time'], utc=True)
        df = df[(df['_time'] >= BD_END) & (df['metric'].isin(METRICS_NET))].copy()
        if df.empty:
            del df; gc.collect()
            continue

        df['window'] = df['_time'].dt.floor('30min')

        grp = df.groupby(['source_agent','cloud_provider','region','metric','window'])['value']
        stats = grp.agg(
            p50    = lambda x: float(np.percentile(x, 50)),
            p95    = lambda x: float(np.percentile(x, 95)),
            mad    = lambda x: float(np.median(np.abs(x - np.median(x)))),
            mean   = 'mean',
            std    = 'std',
            n_samp = 'count',
        ).reset_index()

        net_records.append(stats)
        del df, stats; gc.collect()
        total_rows = sum(len(r) for r in net_records)
        print(f'  [{fi+1}/{len(files)}] {os.path.basename(f)} — acum. {total_rows:,} filas')

    print(f'  ✅ {prov.upper()} listo')

net_df = pd.concat(net_records, ignore_index=True)
del net_records; gc.collect()

print(f'\nnet_df shape: {net_df.shape}')
print(f'Columnas    : {list(net_df.columns)}')


[AWS] 13 archivos...
  [2/13] te_aws_raw_may_w2.parquet — acum. 406,777 filas
  [3/13] te_aws_raw_may_w3.parquet — acum. 814,720 filas
  [4/13] te_aws_raw_may_w4.parquet — acum. 1,215,544 filas
  [5/13] te_aws_raw_may_w5.parquet — acum. 1,448,822 filas
  [6/13] te_aws_raw_w1.parquet — acum. 1,854,630 filas
  [7/13] te_aws_raw_w2.parquet — acum. 2,259,839 filas
  [8/13] te_aws_raw_w3.parquet — acum. 2,665,934 filas
  [9/13] te_aws_raw_w4.parquet — acum. 3,072,824 filas
  [10/13] te_aws_raw_w5.parquet — acum. 3,247,782 filas
  [11/13] te_aws_raw_w6.parquet — acum. 3,655,857 filas
  [12/13] te_aws_raw_w7.parquet — acum. 4,064,054 filas
  [13/13] te_aws_raw_w8.parquet — acum. 4,413,954 filas
  ✅ AWS listo

[AZURE] 13 archivos...
  [2/13] te_azure_raw_may_w2.parquet — acum. 4,730,340 filas
  [3/13] te_azure_raw_may_w3.parquet — acum. 5,047,629 filas
  [4/13] te_azure_raw_may_w4.parquet — acum. 5,365,065 filas
  [5/13] te_azure_raw_may_w5.parquet — acum. 5,546,505 filas
  [6/13] te_azure_ra

In [4]:
net_df['metric_short'] = net_df['metric'].map(METRIC_SHORT)

base_lookup = baseline_df[['source_agent','cloud_provider','region','metric','p95','mad']].copy()
base_lookup = base_lookup.rename(columns={'p95': 'base_p95', 'mad': 'base_mad'})

net_df = net_df.merge(base_lookup, on=['source_agent','cloud_provider','region','metric'], how='left')

net_df['delta_p95'] = (net_df['p95'] - net_df['base_p95']) / (net_df['base_p95'] + EPS)
net_df['delta_mad'] = (net_df['mad'] - net_df['base_mad']) / (net_df['base_mad'] + EPS)

index_cols = ['source_agent','cloud_provider','region','window']
value_cols = ['p50','p95','mad','mean','std','n_samp','delta_p95','delta_mad']

wide_df = net_df.pivot_table(
    index=index_cols, columns='metric_short', values=value_cols, aggfunc='first',
)
wide_df.columns = [f'{stat}_{met}' for stat, met in wide_df.columns]
wide_df = wide_df.reset_index()
wide_df['window'] = pd.to_datetime(wide_df['window'], utc=True)

del net_df, base_lookup; gc.collect()

print(f'wide_df shape: {wide_df.shape}')
print(f'Columnas ({len(wide_df.columns)}):')
for c in sorted(wide_df.columns):
    print(f'  {c}')

wide_df shape: (3763917, 28)
Columnas (28):
  cloud_provider
  delta_mad_jitter
  delta_mad_lat
  delta_mad_loss
  delta_p95_jitter
  delta_p95_lat
  delta_p95_loss
  mad_jitter
  mad_lat
  mad_loss
  mean_jitter
  mean_lat
  mean_loss
  n_samp_jitter
  n_samp_lat
  n_samp_loss
  p50_jitter
  p50_lat
  p50_loss
  p95_jitter
  p95_lat
  p95_loss
  region
  source_agent
  std_jitter
  std_lat
  std_loss
  window


In [5]:

INDEX_COLS = ['source_agent', 'cloud_provider', 'region', 'window']

LAG_COLS = [c for c in wide_df.columns
            if c not in INDEX_COLS
            and not c.startswith('n_samp')]

print(f'Columnas a laguear: {len(LAG_COLS)}')
print('  ' + ', '.join(sorted(LAG_COLS)))

lag1_src = wide_df[INDEX_COLS + LAG_COLS].copy()
lag1_src['window'] = lag1_src['window'] + WINDOW
lag1_src = lag1_src.rename(columns={c: f'lag1_{c}' for c in LAG_COLS})

lag2_src = wide_df[INDEX_COLS + LAG_COLS].copy()
lag2_src['window'] = lag2_src['window'] + 2 * WINDOW
lag2_src = lag2_src.rename(columns={c: f'lag2_{c}' for c in LAG_COLS})

lag48_src = wide_df[INDEX_COLS + LAG_COLS].copy()
lag48_src['window'] = lag48_src['window'] + 48 * WINDOW
lag48_src = lag48_src.rename(columns={c: f'lag48_{c}' for c in LAG_COLS})

lag336_src = wide_df[INDEX_COLS + LAG_COLS].copy()
lag336_src['window'] = lag336_src['window'] + 336 * WINDOW
lag336_src = lag336_src.rename(columns={c: f'lag336_{c}' for c in LAG_COLS})

wide_v4 = (wide_df
           .merge(lag1_src,   on=INDEX_COLS, how='left')
           .merge(lag2_src,   on=INDEX_COLS, how='left')
           .merge(lag48_src,  on=INDEX_COLS, how='left')
           .merge(lag336_src, on=INDEX_COLS, how='left'))

del lag1_src, lag2_src, lag48_src, lag336_src; gc.collect()

n_lag1_nan   = wide_v4[[f'lag1_{c}'   for c in LAG_COLS]].isnull().any(axis=1).sum()
n_lag2_nan   = wide_v4[[f'lag2_{c}'   for c in LAG_COLS]].isnull().any(axis=1).sum()
n_lag48_nan  = wide_v4[[f'lag48_{c}'  for c in LAG_COLS]].isnull().any(axis=1).sum()
n_lag336_nan = wide_v4[[f'lag336_{c}' for c in LAG_COLS]].isnull().any(axis=1).sum()
n_total      = len(wide_v4)

print(f'\nwide_v4 shape : {wide_v4.shape}')
print(f'  Features base        : {len(wide_df.columns) - len(INDEX_COLS)}')
print(f'  Lag t-1  cols        : {len(LAG_COLS)}')
print(f'  Lag t-2  cols        : {len(LAG_COLS)}')
print(f'  Lag t-48  cols       : {len(LAG_COLS)}  [test hipotesis 24h]')
print(f'  Lag t-336 cols       : {len(LAG_COLS)}  [test hipotesis 7d]')
print(f'\nNaN lag t-1   : {n_lag1_nan:,} / {n_total:,} ({100*n_lag1_nan/n_total:.1f}%)')
print(f'NaN lag t-2   : {n_lag2_nan:,} / {n_total:,} ({100*n_lag2_nan/n_total:.1f}%)')
print(f'NaN lag t-48  : {n_lag48_nan:,} / {n_total:,} ({100*n_lag48_nan/n_total:.1f}%)  (primeras 24h por ruta)')
print(f'NaN lag t-336 : {n_lag336_nan:,} / {n_total:,} ({100*n_lag336_nan/n_total:.1f}%)  (primeros 7d por ruta)')
print(f'(NaN esperado: XGBoost/RF manejan NaN nativamente)')
print(f'\nW_v4 = {W_v4}min >= H = {H}min  [t-1/t-2]  ✅  Hyndman & Athanasopoulos (2021) §8.7')
print(f'Lags t-48/t-336 : TEST DE HIPOTESIS — MDI post-entrenamiento decide')


Columnas a laguear: 21
  delta_mad_jitter, delta_mad_lat, delta_mad_loss, delta_p95_jitter, delta_p95_lat, delta_p95_loss, mad_jitter, mad_lat, mad_loss, mean_jitter, mean_lat, mean_loss, p50_jitter, p50_lat, p50_loss, p95_jitter, p95_lat, p95_loss, std_jitter, std_lat, std_loss

wide_v4 shape : (3763917, 112)
  Features base        : 24
  Lag t-1  cols        : 21
  Lag t-2  cols        : 21
  Lag t-48  cols       : 21  [test hipotesis 24h]
  Lag t-336 cols       : 21  [test hipotesis 7d]

NaN lag t-1   : 2,715 / 3,763,917 (0.1%)
NaN lag t-2   : 4,069 / 3,763,917 (0.1%)
NaN lag t-48  : 58,302 / 3,763,917 (1.5%)  (primeras 24h por ruta)
NaN lag t-336 : 358,570 / 3,763,917 (9.5%)  (primeros 7d por ruta)
(NaN esperado: XGBoost/RF manejan NaN nativamente)

W_v4 = 90min >= H = 60min  [t-1/t-2]  ✅  Hyndman & Athanasopoulos (2021) §8.7
Lags t-48/t-336 : TEST DE HIPOTESIS — MDI post-entrenamiento decide


In [6]:

wide_v4['hour']     = wide_v4['window'].dt.hour
wide_v4['hour_sin'] = np.sin(2 * np.pi * wide_v4['hour'] / 24)
wide_v4['hour_cos'] = np.cos(2 * np.pi * wide_v4['hour'] / 24)

wide_v4['dow']      = wide_v4['window'].dt.dayofweek
wide_v4['dow_sin']  = np.sin(2 * np.pi * wide_v4['dow'] / 7)
wide_v4['dow_cos']  = np.cos(2 * np.pi * wide_v4['dow'] / 7)

wide_v4['is_weekend'] = (wide_v4['dow'] >= 5).astype(np.int8)

wide_v4.drop(columns=['hour', 'dow'], inplace=True)

TEMPORAL_COLS = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']

print(f'Temporal features agregadas: {TEMPORAL_COLS}')
print(f'wide_v4 shape tras HOD+DOW : {wide_v4.shape}')
print()
print('=== Estadisticas descriptivas (temporal features) ===')
print(wide_v4[TEMPORAL_COLS].describe().round(4).to_string())
print()
print('=== Distribucion is_weekend ===')
vc = wide_v4['is_weekend'].value_counts().sort_index()
for v, cnt in vc.items():
    label = 'fin_semana' if v == 1 else 'entre_semana'
    print(f'  {v} ({label}): {cnt:,}  ({100*cnt/len(wide_v4):.1f}%)')
print()
print('=== Rango de valores sin/cos ===')
for col in ['hour_sin','hour_cos','dow_sin','dow_cos']:
    print(f'  {col}: [{wide_v4[col].min():.4f}, {wide_v4[col].max():.4f}]')
print(f'\nW_v4 final = {W_v4}min | Features totales en wide_v4: {len(wide_v4.columns) - 4} '
      f'(excl. index cols source_agent, cloud_provider, region, window)')


Temporal features agregadas: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']
wide_v4 shape tras HOD+DOW : (3763917, 117)

=== Estadisticas descriptivas (temporal features) ===
           hour_sin      hour_cos       dow_sin       dow_cos    is_weekend
count  3.763917e+06  3.763917e+06  3.763917e+06  3.763917e+06  3.763917e+06
mean  -0.000000e+00 -4.000000e-04 -5.900000e-03  1.240000e-02  2.896000e-01
std    7.071000e-01  7.071000e-01  7.100000e-01  7.041000e-01  4.536000e-01
min   -1.000000e+00 -1.000000e+00 -9.749000e-01 -9.010000e-01  0.000000e+00
25%   -7.071000e-01 -7.071000e-01 -7.818000e-01 -9.010000e-01  0.000000e+00
50%    0.000000e+00 -0.000000e+00  0.000000e+00 -2.225000e-01  0.000000e+00
75%    7.071000e-01  7.071000e-01  7.818000e-01  6.235000e-01  1.000000e+00
max    1.000000e+00  1.000000e+00  9.749000e-01  1.000000e+00  1.000000e+00

=== Distribucion is_weekend ===
  0 (entre_semana): 2,674,008  (71.0%)
  1 (fin_semana): 1,089,909  (29.0%)

=== Rango de valo

In [7]:
BGP_METRIC_SHORT = {
    'bgp.path_changes.count': 'bgp_path_changes',
    'bgp.updates.count':      'bgp_updates',
    'bgp.reachability':       'bgp_reachability',
}

bgp_records = []

for prov, files in PROVIDER_MAP.items():
    print(f'[{prov.upper()}] BGP...')
    for f in files:
        try:
            tbl = pq.read_table(f, columns=['_time','cloud_provider','region','metric','value'])
        except Exception as e:
            print(f'  ⚠️  {os.path.basename(f)}: {e}')
            continue

        df = tbl.to_pandas(); del tbl; gc.collect()
        df['_time'] = pd.to_datetime(df['_time'], utc=True)
        df = df[(df['_time'] >= BD_END) & (df['metric'].isin(METRICS_BGP))].copy()
        if df.empty:
            del df; gc.collect()
            continue

        df['window'] = df['_time'].dt.floor('30min')
        grp      = df.groupby(['cloud_provider','region','metric','window'])['value']
        agg_sum  = grp.sum().rename('val_sum')
        agg_mean = grp.mean().rename('val_mean')
        agg_df   = pd.concat([agg_sum, agg_mean], axis=1).reset_index()
        bgp_records.append(agg_df)
        del df, agg_df; gc.collect()

if bgp_records:
    bgp_df = pd.concat(bgp_records, ignore_index=True)
    del bgp_records; gc.collect()

    bgp_df['bgp_val'] = bgp_df.apply(
        lambda r: r['val_mean'] if 'reachability' in r['metric'] else r['val_sum'], axis=1)
    bgp_df['metric_short'] = bgp_df['metric'].map(BGP_METRIC_SHORT)

    bgp_wide = bgp_df.pivot_table(
        index=['cloud_provider','region','window'],
        columns='metric_short', values='bgp_val', aggfunc='first',
    ).reset_index()
    bgp_wide['window'] = pd.to_datetime(bgp_wide['window'], utc=True)

    print(f'BGP features shape: {bgp_wide.shape}')
    print(f'Columnas BGP: {list(bgp_wide.columns)}')
else:
    print('⚠️  Sin datos BGP — se omite esta feature')
    bgp_wide = None

[AWS] BGP...
[AZURE] BGP...
[GCP] BGP...
BGP features shape: (78277, 6)
Columnas BGP: ['cloud_provider', 'region', 'window', 'bgp_path_changes', 'bgp_reachability', 'bgp_updates']


In [8]:
feat_df = wide_v4.merge(
    labeled_df[['source_agent','cloud_provider','region','window','label_ventana','split']],
    on=['source_agent','cloud_provider','region','window'],
    how='inner'
)

if bgp_wide is not None:
    feat_df = feat_df.merge(bgp_wide, on=['cloud_provider','region','window'], how='left')

print(f'Feature dataset v4 shape: {feat_df.shape}')
print(f'Columnas ({len(feat_df.columns)}):')
for c in sorted(feat_df.columns):
    print(f'  {c}')

print(f'\n=== Splits ===')
print(feat_df['split'].value_counts())

print(f'\n=== Distribucion de labels ===')
labels_map = {0:'Normal', 1:'Baja', 2:'Media', 3:'Alta'}
dist = feat_df['label_ventana'].value_counts(normalize=True).sort_index() * 100
for k, pct in dist.items():
    print(f'  {labels_map[k]:8s} ({k}): {pct:.1f}%')

print(f'\n=== Nulos por columna ===')
nulls = feat_df.isnull().sum()
nulls = nulls[nulls > 0]
if not nulls.empty:
    print(nulls.sort_values(ascending=False).to_string())
else:
    print('  Sin nulos')

print(f'\n=== Cobertura ===')
for split in ['train', 'val', 'test']:
    n_label = (labeled_df['split'] == split).sum()
    n_feat  = (feat_df['split'] == split).sum()
    pct = 100 * n_feat / n_label if n_label > 0 else 0
    print(f'  {split:5s}: labels={n_label:>10,}  features={n_feat:>10,}  ({pct:.1f}%)')

n_lag_cols = len([c for c in feat_df.columns if c.startswith('lag')])
n_base_cols = len(feat_df.columns) - n_lag_cols - len(['label_ventana','split'])
print(f'\n=== Composicion de features ===')
n_temporal_cols = len([c for c in feat_df.columns if c in ['hour_sin','hour_cos','dow_sin','dow_cos','is_weekend']])
print(f'  Base (v3)    : {n_base_cols} columnas')
print(f'  Lag t-1+t-2  : {n_lag_cols} columnas')
print(f'  Temporal HOD : {n_temporal_cols} columnas')
print(f'  Total modelo : {n_lag_cols + n_base_cols + n_temporal_cols} features')


Feature dataset v4 shape: (3763917, 122)
Columnas (122):
  bgp_path_changes
  bgp_reachability
  bgp_updates
  cloud_provider
  delta_mad_jitter
  delta_mad_lat
  delta_mad_loss
  delta_p95_jitter
  delta_p95_lat
  delta_p95_loss
  dow_cos
  dow_sin
  hour_cos
  hour_sin
  is_weekend
  label_ventana
  lag1_delta_mad_jitter
  lag1_delta_mad_lat
  lag1_delta_mad_loss
  lag1_delta_p95_jitter
  lag1_delta_p95_lat
  lag1_delta_p95_loss
  lag1_mad_jitter
  lag1_mad_lat
  lag1_mad_loss
  lag1_mean_jitter
  lag1_mean_lat
  lag1_mean_loss
  lag1_p50_jitter
  lag1_p50_lat
  lag1_p50_loss
  lag1_p95_jitter
  lag1_p95_lat
  lag1_p95_loss
  lag1_std_jitter
  lag1_std_lat
  lag1_std_loss
  lag2_delta_mad_jitter
  lag2_delta_mad_lat
  lag2_delta_mad_loss
  lag2_delta_p95_jitter
  lag2_delta_p95_lat
  lag2_delta_p95_loss
  lag2_mad_jitter
  lag2_mad_lat
  lag2_mad_loss
  lag2_mean_jitter
  lag2_mean_lat
  lag2_mean_loss
  lag2_p50_jitter
  lag2_p50_lat
  lag2_p50_loss
  lag2_p95_jitter
  lag2_p95_lat


In [9]:
TEMPORAL_COLS = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']

feat_df.to_parquet(OUT_FEAT, index=False)
print(f'✅ Guardado: {OUT_FEAT}')
print(f'   Shape    : {feat_df.shape}')
print(f'   Columnas : {len(feat_df.columns)}')
n_lag1   = len([c for c in feat_df.columns if c.startswith('lag1_')])
n_lag2   = len([c for c in feat_df.columns if c.startswith('lag2_')])
n_lag48  = len([c for c in feat_df.columns if c.startswith('lag48_')])
n_lag336 = len([c for c in feat_df.columns if c.startswith('lag336_')])
n_temp   = len([c for c in feat_df.columns if c in TEMPORAL_COLS])
n_bgp    = len([c for c in feat_df.columns if c.startswith('bgp_')])
print(f'   Lag t-1   : {n_lag1}  features')
print(f'   Lag t-2   : {n_lag2}  features')
print(f'   Lag t-48  : {n_lag48}  features  [test hipotesis 24h]')
print(f'   Lag t-336 : {n_lag336}  features  [test hipotesis 7d]')
print(f'   Temporal  : {n_temp} features  ({TEMPORAL_COLS})')
print(f'   BGP       : {n_bgp} features')
print(f'   Splits    : {feat_df["split"].value_counts().to_dict()}')
print(f'\nPrimeras 3 filas:')
print(feat_df.head(3).T)


✅ Guardado: /srv/work/te_features_v4.parquet
   Shape    : (3763917, 122)
   Columnas : 122
   Lag t-1   : 21  features
   Lag t-2   : 21  features
   Lag t-48  : 21  features  [test hipotesis 24h]
   Lag t-336 : 21  features  [test hipotesis 7d]
   Temporal  : 5 features  (['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend'])
   BGP       : 3 features
   Splits    : {'train': 2125778, 'test': 993449, 'val': 644690}

Primeras 3 filas:
                                             0                             1  \
source_agent      Belo Horizonte, Brazil (TIM)  Belo Horizonte, Brazil (TIM)   
cloud_provider                             AWS                           AWS   
region                      Chicago-Local-Zone            Chicago-Local-Zone   
window               2026-05-08 00:00:00+00:00     2026-05-08 00:30:00+00:00   
delta_mad_jitter                      -0.47649                     -0.097561   
...                                        ...                           